# Student Presentation Scheduler & Group Optimizer
I've designed this tool to conduct a fair 2-round group presentation schedule for **any number of students** — change one number (`N_STUDENTS`) each term and re-run.

The algorithm distributes students into two rounds of presentations, split across a small number of class days, optimizing for peer diversity and scheduling fairness.

---

## 📋 Problem Parameters & Constraints

### 1. Structural Constraints
* **Total Population ($N$):** configurable — set by `N_STUDENTS` below.
* **Total Rounds:** 2 rounds (Round A and Round B).
* **Group Configuration:** the number of groups per round and their sizes are *computed* from $N$ to stay close to `TARGET_GROUP_SIZE_RANGE` (default 4–5 students per group) — not hardcoded.
* **Total Presentations:** $2 \times$ (groups per round), split as evenly as possible across `NUM_CLASS_DAYS` class days.

### 2. Peer Diversity Constraint (Zero-Overlap)
* Let $G_{Ax}$ be a group in Round A and $G_{By}$ be a group in Round B.
* The algorithm guarantees that for all pairs $(x, y)$:
$$\lvert G_{Ax} \cap G_{By} \rvert \le 1$$
* **Meaning:** No two students will ever work together in both Round A and Round B. Every student gets a completely fresh set of team members for their second presentation.

### 3. Scheduling Constraint (No Same-Day Conflicts)
* No student is permitted to present twice on the same class day.

---

## 🛠️ Implementation & Validation

In [2]:
import math
import pandas as pd

# ============================================================
# CONFIGURATION - the only section a professor needs to touch each term
# ============================================================
N_STUDENTS = 26                    # <- update this each term (e.g. 30 next term)
TARGET_GROUP_SIZE_RANGE = (4, 5)   # a sane target, not a hard requirement
NUM_CLASS_DAYS = 3
MIN_GROUP_SIZE = 2                 # a "group" of 1 student isn't a group
K_SEARCH_SPAN = 12                 # how many group-counts to try before giving up
DAY_SEARCH_NODE_LIMIT = 50_000     # safety valve so the day-search can't hang

student_roster = [f"Student_{i}" for i in range(1, N_STUDENTS + 1)]
# Or supply real names instead: student_roster = ["Alice", "Bob", ...]


# ============================================================
# GROUP CONSTRUCTION - a pure grid/transpose formula, no search.
# This is the part that is *not* the hard "social golfer problem": for
# exactly 2 rounds, laying students into a grid (Round A = rows, Round B
# = columns) guarantees zero-overlap for ANY N, because a row and a
# column can only ever share one cell.
# ============================================================

def partition_students(roster, k):
    """Split roster into k groups whose sizes differ by at most 1."""
    n = len(roster)
    base, extra = divmod(n, k)
    groups, start = [], 0
    for i in range(k):
        size = base + (1 if i < extra else 0)
        groups.append(roster[start:start + size])
        start += size
    return groups


def build_round_b_groups(round_a_groups, k, offset=0):
    """
    Deal Round A's students round-robin into k Round B groups (the
    "columns" of the grid). As long as no Round A group is larger than k,
    no two students from the same Round A group can land in the same
    Round B group - zero-overlap holds by construction, for any N.
    """
    flat = [student for group in round_a_groups for student in group]
    round_b = [[] for _ in range(k)]
    for i, student in enumerate(flat):
        round_b[(i + offset) % k].append(student)
    return round_b


# ============================================================
# DAY ASSIGNMENT - the one genuinely combinatorial step here, but a tiny
# one: a dozen-ish groups and a handful of days, solved instantly by
# backtracking search with constraint propagation. No external SAT/CP
# solver is needed - that machinery is for the *many-round* version of
# the social golfer problem, which this isn't.
# ============================================================

def assign_days(round_a_groups, round_b_groups, num_days, node_limit=DAY_SEARCH_NODE_LIMIT):
    """
    Assign each Round A/B group a class day (1..num_days) so no student
    presents twice on the same day, keeping days as balanced as possible.
    Returns None if no valid assignment is found within the node budget.
    """
    k = len(round_a_groups)
    labels = [f"A{i+1}" for i in range(k)] + [f"B{i+1}" for i in range(k)]

    # Two groups conflict if they share a student - they can't share a day.
    adjacency = {label: set() for label in labels}
    for i, group_a in enumerate(round_a_groups):
        for j, group_b in enumerate(round_b_groups):
            if set(group_a) & set(group_b):
                adjacency[f"A{i+1}"].add(f"B{j+1}")
                adjacency[f"B{j+1}"].add(f"A{i+1}")

    total_groups = 2 * k
    day_capacity = math.ceil(total_groups / num_days)
    days = list(range(1, num_days + 1))
    domains = {label: set(days) for label in labels}
    assignment = {}
    day_counts = {d: 0 for d in days}
    nodes_visited = [0]

    def most_constrained_label():
        unassigned = [l for l in labels if l not in assignment]
        return min(unassigned, key=lambda l: len(domains[l]))

    def backtrack():
        nodes_visited[0] += 1
        if nodes_visited[0] > node_limit:
            raise TimeoutError
        if len(assignment) == len(labels):
            return True
        label = most_constrained_label()
        if not domains[label]:
            return False
        for day in sorted(domains[label], key=lambda d: day_counts[d]):
            if day_counts[day] >= day_capacity:
                continue
            assignment[label] = day
            day_counts[day] += 1
            narrowed = []
            dead_end = False
            for neighbor in adjacency[label]:
                if neighbor not in assignment and day in domains[neighbor]:
                    domains[neighbor].discard(day)
                    narrowed.append(neighbor)
                    if not domains[neighbor]:
                        dead_end = True
                        break
            if not dead_end and backtrack():
                return True
            for neighbor in narrowed:
                domains[neighbor].add(day)
            day_counts[day] -= 1
            del assignment[label]
        return False

    try:
        return dict(assignment) if backtrack() else None
    except TimeoutError:
        return None


# ============================================================
# SCHEDULE BUILDER - ties the two steps together, searching over group
# counts (and, as a fallback, Round B dealing offsets) until both
# constraints are satisfied simultaneously.
# ============================================================

def build_schedule(roster, size_range=TARGET_GROUP_SIZE_RANGE, num_days=NUM_CLASS_DAYS,
                    k_search_span=K_SEARCH_SPAN):
    n = len(roster)
    lo, hi = size_range
    k_min = max(2, math.ceil(n / hi))

    for k in range(k_min, k_min + k_search_span):
        round_a = partition_students(roster, k)
        sizes = [len(g) for g in round_a]
        if max(sizes) > k or min(sizes) < MIN_GROUP_SIZE:
            continue  # this K can't stay overlap-free, or produces degenerate groups
        for offset in range(k):
            round_b = build_round_b_groups(round_a, k, offset=offset)
            day_assignment = assign_days(round_a, round_b, num_days)
            if day_assignment is not None:
                return round_a, round_b, day_assignment, k

    raise ValueError(
        f"Could not build a conflict-free {num_days}-day schedule for N={n} students "
        f"within {k_search_span} group-count attempts (starting at {k_min} groups per round). "
        f"Try increasing NUM_CLASS_DAYS, or double-check N isn't too small for two "
        f"rounds of non-overlapping groups."
    )


# ============================================================
# VALIDATION - checks BOTH constraints
# ============================================================

def validate_constraints(round_a_groups, round_b_groups, day_assignment):
    valid = True

    for i, group_a in enumerate(round_a_groups):
        for j, group_b in enumerate(round_b_groups):
            overlap = set(group_a) & set(group_b)
            if len(overlap) > 1:
                print(f"[!] Overlap Error: A{i+1} and B{j+1} share {len(overlap)} members: {overlap}")
                valid = False

    student_day_a = {s: day_assignment[f"A{i+1}"] for i, g in enumerate(round_a_groups) for s in g}
    student_day_b = {s: day_assignment[f"B{i+1}"] for i, g in enumerate(round_b_groups) for s in g}
    for student, day_a in student_day_a.items():
        if day_a == student_day_b[student]:
            print(f"[!] Same-Day Conflict: {student} is scheduled twice on day {day_a}")
            valid = False

    if valid:
        print("[-] Validation Passed: 0 overlapping group members detected across all rounds.")
        print("[-] Validation Passed: 0 same-day presentation conflicts detected.\n")

    return valid


# ==========================================
# Execution
# ==========================================

round_a, round_b, day_assignment, num_groups = build_schedule(student_roster)

print("--- Configuration Summary ---")
print(f"Students: {len(student_roster)} | Groups per round: {num_groups} | Class days: {NUM_CLASS_DAYS}")
print(f"Round A group sizes: {[len(g) for g in round_a]}")
print(f"Round B group sizes: {[len(g) for g in round_b]}\n")

assert validate_constraints(round_a, round_b, day_assignment), \
    "Schedule failed validation - this should never happen; please report it."

# Format for clean output
schedule_data = []
for i, group in enumerate(round_a):
    schedule_data.append({
        "Day": day_assignment[f"A{i+1}"],
        "Class Day": f"Class {day_assignment[f'A{i+1}']}",
        "Round": "A",
        "Group Name": f"A{i+1}",
        "Size": len(group),
        "Members": ", ".join(group),
    })
for i, group in enumerate(round_b):
    schedule_data.append({
        "Day": day_assignment[f"B{i+1}"],
        "Class Day": f"Class {day_assignment[f'B{i+1}']}",
        "Round": "B",
        "Group Name": f"B{i+1}",
        "Size": len(group),
        "Members": ", ".join(group),
    })

df_schedule = pd.DataFrame(schedule_data).sort_values(["Day", "Round", "Group Name"]).drop(columns="Day")
df_schedule = df_schedule.reset_index(drop=True)

print("--- Final Master Schedule ---")
print(df_schedule.to_string(index=False))

# Optional: Export to CSV
# df_schedule.to_csv("presentation_schedule.csv", index=False)


--- Configuration Summary ---
Students: 26 | Groups per round: 6 | Class days: 3
Round A group sizes: [5, 5, 4, 4, 4, 4]
Round B group sizes: [5, 5, 4, 4, 4, 4]

[-] Validation Passed: 0 overlapping group members detected across all rounds.
[-] Validation Passed: 0 same-day presentation conflicts detected.

--- Final Master Schedule ---
Class Day Round Group Name  Size                                                  Members
  Class 1     A         A1     5    Student_1, Student_2, Student_3, Student_4, Student_5
  Class 1     A         A2     5   Student_6, Student_7, Student_8, Student_9, Student_10
  Class 1     A         A4     4           Student_15, Student_16, Student_17, Student_18
  Class 1     A         A5     4           Student_19, Student_20, Student_21, Student_22
  Class 2     B         B1     5 Student_1, Student_7, Student_13, Student_19, Student_25
  Class 2     B         B2     5 Student_2, Student_8, Student_14, Student_20, Student_26
  Class 2     B         B5     